In [3]:
import torch
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import os

DATA_PATH = "../data/train_data.tsv"
TENSORS_PATH = "dataset_tensor_decoder_only.pt"
VOCAB_SIZE = 30000

def train_tokenizer_and_process():
    if not os.path.exists("tokenizer_decoder_only.json"):
        print("Training Tokenizer...")
        tokenizer = Tokenizer(BPE(unk_token="<unk>"))
        tokenizer.pre_tokenizer = Whitespace()
        trainer = BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<pad>", "<sos>", "<eos>", "<unk>", "<sep>"], min_frequency=2)
        
        def data_gen():
            with open(DATA_PATH, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) >= 4:
                        yield parts[1] # Eng
                        yield parts[3] # Rus

        tokenizer.train_from_iterator(data_gen(), trainer)
        tokenizer.save("tokenizer_decoder_only.json")
    else:
        print("Loading Tokenizer...")
        tokenizer = Tokenizer.from_file("tokenizer_decoder_only.json")

    print("Tokenizing dataset for Decoder-Only...")

    src_seqs = []
    tgt_seqs = []
    
    sos_id = tokenizer.token_to_id("<sos>")
    eos_id = tokenizer.token_to_id("<eos>")

    with open(DATA_PATH, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i % 100000 == 0: print(f"Processed {i} lines")
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                src_encoded = tokenizer.encode(parts[1]).ids
                tgt_encoded = tokenizer.encode(parts[3]).ids
                
                src_seqs.append(torch.tensor([sos_id] + src_encoded + [eos_id], dtype=torch.int16))
                tgt_seqs.append(torch.tensor([sos_id] + tgt_encoded + [eos_id], dtype=torch.int16))

    print(f"Saving tensors to {TENSORS_PATH}...")
    torch.save({"src": src_seqs, "tgt": tgt_seqs}, TENSORS_PATH)
    print("Done! Ready for training.")

if __name__ == "__main__":
    train_tokenizer_and_process()


Loading Tokenizer...
Tokenizing dataset for Decoder-Only...


FileNotFoundError: [Errno 2] No such file or directory: './data/train_data.tsv'

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tokenizers import Tokenizer
from torch.cuda.amp import autocast, GradScaler 
from tqdm.auto import tqdm
import os
import csv
import math

from model_decoder_only import TransformerDecoderOnly

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 128      
LR = 0.0002
EPOCHS = 20
D_MODEL = 256         
N_LAYERS = 4          
CHECKPOINT_PATH = "checkpoint_decoder_only.pt"
METRICS_PATH = "metrics_decoder_only.csv"
RESUME = False        
TENSORS_PATH = "dataset_tensor_decoder_only.pt"

class DecoderOnlyTranslationDataset(Dataset):
    def __init__(self, tensors_path, sep_token_id):
        print("Loading tensors into RAM...")
        data = torch.load(tensors_path)
        self.src_list = data["src"]
        self.tgt_list = data["tgt"]
        self.sep_id = sep_token_id
        print(f"Loaded {len(self.src_list)} examples.")

    def __len__(self):
        return len(self.src_list)

    def __getitem__(self, idx):
        src = self.src_list[idx]
        tgt = self.tgt_list[idx]
        
        src_part = src[:-1]
        tgt_part = tgt[1:]  
        
        sep_tensor = torch.tensor([self.sep_id], dtype=torch.int16)
        
        full_seq = torch.cat([src_part, sep_tensor, tgt_part])
        
        return full_seq.long()

def collate_batch(batch, pad_idx):
    batch_padded = torch.nn.utils.rnn.pad_sequence(batch, padding_value=pad_idx, batch_first=True)
    return batch_padded

def save_metrics(path, epoch, train_loss, train_ppl, val_loss, val_ppl):
    file_exists = os.path.isfile(path)
    with open(path, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Epoch', 'Train Loss', 'Train PPL', 'Val Loss', 'Val PPL'])
        writer.writerow([epoch, f"{train_loss:.4f}", f"{train_ppl:.4f}", f"{val_loss:.4f}", f"{val_ppl:.4f}"])

tokenizer = Tokenizer.from_file("tokenizer_decoder_only.json")
PAD_IDX = tokenizer.token_to_id("<pad>")
SEP_IDX = tokenizer.token_to_id("<sep>")
if SEP_IDX is None:
    print("Warning: <sep> token not found using <eos> as separator")
    SEP_IDX = tokenizer.token_to_id("<eos>")

VOCAB_SIZE = tokenizer.get_vocab_size()

dataset = DecoderOnlyTranslationDataset(TENSORS_PATH, SEP_IDX)
train_size = int(0.95 * len(dataset))
val_size = len(dataset) - train_size
train_data, val_data = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, 
                          collate_fn=lambda x: collate_batch(x, PAD_IDX), num_workers=0, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, 
                        collate_fn=lambda x: collate_batch(x, PAD_IDX), num_workers=0)

model = TransformerDecoderOnly(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_layer=N_LAYERS,
    n_head=8,
    d_head=D_MODEL // 8, 
    d_ff=D_MODEL * 4,
    max_len=2000,
    dropout=0.1,
    pad_idx=PAD_IDX
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9)
scaler = GradScaler()
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
start_epoch = 0

if RESUME and os.path.exists(CHECKPOINT_PATH):
    print("Resuming from checkpoint...")
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resuming from epoch {start_epoch}")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    epoch_loss = 0
    
    for batch in loop:
        batch = batch.to(DEVICE)
        
        input_seq = batch[:, :-1]
        target_seq = batch[:, 1:]
        
        optimizer.zero_grad()
        
        with autocast():
            logits = model(input_seq)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), target_seq.reshape(-1))
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    
    avg_loss = epoch_loss / len(train_loader)
    train_ppl = math.exp(min(avg_loss, 100))
    print(f"Epoch {epoch+1} Train Loss: {avg_loss:.4f} | PPL: {train_ppl:.4f}")
    
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss,
    }, CHECKPOINT_PATH)
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            input_seq = batch[:, :-1]
            target_seq = batch[:, 1:]
            
            with autocast():
                logits = model(input_seq)
                loss = criterion(logits.reshape(-1, VOCAB_SIZE), target_seq.reshape(-1))
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_ppl = math.exp(min(avg_val_loss, 100))
    print(f"Val Loss: {avg_val_loss:.4f} | Val PPL: {val_ppl:.4f}")
    
    save_metrics(METRICS_PATH, epoch+1, avg_loss, train_ppl, avg_val_loss, val_ppl)


C:\Users\amust\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tensors into RAM...


KeyboardInterrupt: 

In [7]:
FINAL_MODEL_PATH = "weights_decoder_only.pt"
print("Saving final model...")
torch.save(model.state_dict(), FINAL_MODEL_PATH)
print("Model Saved")

Saving final model...
Model Saved
